# Notebook-first application walkthrough

**Problem / objective:** Classify documents while exposing confidence, errors and the text features that drive the decision.

**Decision / solution:** Auto-route high-confidence documents and send uncertain or unusual text to human review.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'nlp_document_intelligence'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Auto-route high-confidence documents and send uncertain or unusual text to human review.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# NLP Document Intelligence — Classification & Insight Engine

## Problem and objective
Build a natural-language-processing application that routes incoming documents, extracts category-level language insights and sends low-confidence text to human review rather than forcing an automated decision.


## Dataset and provenance
The project uses scikit-learn's 20 Newsgroups corpus with headers, footers and quoted replies removed to reduce source-specific leakage. It is a public benchmark used to demonstrate transferable text-classification engineering, not a claim that newsgroup posts are enterprise support tickets.


In [ ]:
from run import load_corpus, corpus_audit
x_train, y_train, x_test, y_test, target_names = load_corpus()
corpus_audit(x_train, y_train, target_names)


## NLP modelling, analysis and validation
The application includes text normalisation, TF-IDF word/bigram features, Multinomial Naive Bayes baseline, calibrated linear SVM, macro/weighted F1, accuracy, log loss, confusion-pair analysis, confidence thresholds, category keywords, document-length error slices, persisted inference and parity checks. The full canonical implementation is mirrored into the notebook by the portfolio workflow.


In [ ]:
from run import normalise_text
normalise_text('Contact test@example.com and visit https://example.com for graphics help.')


In [ ]:
from run import build_nb_baseline, build_calibrated_svm
baseline = build_nb_baseline()
candidate = build_calibrated_svm()
baseline, candidate


## Decision use, results and limitations
The model returns a predicted category, calibrated probability and either `auto_route` or `human_review`. Results include confidence-coverage tables, top confusion pairs and category keywords under `results/`. Production use requires domain-specific labels, privacy controls, taxonomy governance, novelty detection and ongoing drift/calibration monitoring.

## Reproducibility
Run `python run.py`. The fitted NLP pipeline is saved under `artifacts/`; tests are in `tests/test_nlp.py`.


## Interview discussion
Be ready to explain TF-IDF, unigram/bigram trade-offs, Naive Bayes versus linear SVM, why calibration matters for routing, why headers/quotes are removed, how confidence coverage changes human-review load, how you inspect category keywords without calling them causal explanations, and how you would detect taxonomy or language drift.


# Deeper exploratory analysis and retained evidence

These direct notebook cells extend the initial EDA with data-quality, scale, relationship, output and error diagnostics. They are intentionally visible here rather than hidden behind project helper functions.


In [ ]:
# Extended data-quality scorecard
if df is not None and len(df):
    quality_rows = []
    for col in df.columns:
        series = df[col]
        row = {
            'feature': col,
            'dtype': str(series.dtype),
            'rows': len(series),
            'missing': int(series.isna().sum()),
            'missing_pct': float(100 * series.isna().mean()),
            'unique': int(series.nunique(dropna=False)),
            'unique_pct': float(100 * series.nunique(dropna=False) / max(len(series), 1)),
        }
        if pd.api.types.is_numeric_dtype(series):
            values = pd.to_numeric(series, errors='coerce').dropna()
            if len(values):
                q1, q3 = values.quantile([0.25, 0.75])
                iqr = q3 - q1
                row.update({
                    'mean': float(values.mean()),
                    'median': float(values.median()),
                    'std': float(values.std()),
                    'p05': float(values.quantile(0.05)),
                    'p95': float(values.quantile(0.95)),
                    'skew': float(values.skew()),
                    'iqr_outliers': int(((values < q1 - 1.5*iqr) | (values > q3 + 1.5*iqr)).sum()),
                })
        quality_rows.append(row)
    deep_quality = pd.DataFrame(quality_rows)
    display(deep_quality.sort_values(['missing_pct','unique'], ascending=[False,False]).head(40))
    if 'iqr_outliers' in deep_quality:
        outlier_view = deep_quality.dropna(subset=['iqr_outliers']).sort_values('iqr_outliers', ascending=False).head(15)
        if len(outlier_view):
            plt.figure(figsize=(10,4))
            plt.bar(outlier_view['feature'], outlier_view['iqr_outliers'])
            plt.title('Potential IQR outliers by feature')
            plt.ylabel('Rows')
            plt.xticks(rotation=60, ha='right')
            plt.tight_layout()
            plt.show()
    card = deep_quality.sort_values('unique', ascending=False).head(20)
    plt.figure(figsize=(10,4))
    plt.bar(card['feature'], card['unique'])
    plt.title('Feature cardinality')
    plt.ylabel('Unique values')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()
    plt.show()
    print('Constant columns:', deep_quality.loc[deep_quality['unique'] <= 1, 'feature'].tolist())
    print('High-missing columns:', deep_quality.loc[deep_quality['missing_pct'] >= 30, 'feature'].tolist())
    print('Possible identifier columns:', deep_quality.loc[deep_quality['unique_pct'] >= 95, 'feature'].tolist()[:20])
else:
    print('Materialise the documented dataset to run the extended data-quality scorecard.')


In [ ]:
# Numeric distributions, spread and strongest pairwise relationships
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:12]
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 5:
            continue
        clipped = values.clip(values.quantile(0.01), values.quantile(0.99))
        plt.figure(figsize=(8,4))
        plt.hist(clipped, bins=35, alpha=0.82)
        plt.axvline(values.median(), linestyle='--', label=f'median={values.median():.3g}')
        plt.axvline(values.mean(), linestyle=':', label=f'mean={values.mean():.3g}')
        plt.title(f'Distribution: {col} (1st–99th percentile)')
        plt.xlabel(col)
        plt.ylabel('Rows')
        plt.legend()
        plt.tight_layout()
        plt.show()
        plt.figure(figsize=(8,3))
        plt.boxplot(values, vert=False, showfliers=True)
        plt.title(f'Spread / outliers: {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        pairs = []
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i+1:]:
                value = corr.loc[left, right]
                if pd.notna(value):
                    pairs.append({'feature_a': left, 'feature_b': right, 'correlation': float(value), 'abs_correlation': float(abs(value))})
        corr_pairs = pd.DataFrame(pairs).sort_values('abs_correlation', ascending=False) if pairs else pd.DataFrame()
        if len(corr_pairs):
            display(corr_pairs.head(20).round(4))
            for _, pair in corr_pairs.head(4).iterrows():
                sample = df[[pair['feature_a'], pair['feature_b']]].dropna()
                if len(sample) > 3000:
                    sample = sample.sample(3000, random_state=42)
                plt.figure(figsize=(7,5))
                plt.scatter(sample[pair['feature_a']], sample[pair['feature_b']], alpha=0.30, s=16)
                plt.xlabel(pair['feature_a'])
                plt.ylabel(pair['feature_b'])
                plt.title(f"{pair['feature_a']} vs {pair['feature_b']} (r={pair['correlation']:.2f})")
                plt.tight_layout()
                plt.show()
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 20][:8]
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(20)
        shares = 100 * counts / counts.sum()
        display(pd.DataFrame({'rows': counts, 'share_pct': shares.round(2)}))
        plt.figure(figsize=(8,4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Category balance: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()
else:
    print('Materialise the documented dataset to run distribution diagnostics.')


In [ ]:
# Temporal coverage where date/time fields exist
if df is not None and len(df):
    time_cols = [c for c in df.columns if any(token in str(c).lower() for token in ('date','time','timestamp','datetime'))]
    print('Date/time candidates:', time_cols[:10])
    for col in time_cols[:4]:
        converted = pd.to_datetime(df[col], errors='coerce')
        valid = converted.dropna()
        if len(valid) >= max(10, int(0.25*len(df))):
            print(col, 'range:', valid.min(), '→', valid.max())
            monthly = valid.dt.to_period('M').value_counts().sort_index()
            if len(monthly) > 1:
                plt.figure(figsize=(10,4))
                plt.plot(monthly.index.astype(str), monthly.values, marker='o')
                plt.title(f'Rows over time: {col}')
                plt.ylabel('Rows')
                plt.xticks(rotation=70, ha='right')
                plt.tight_layout()
                plt.show()


## Retained outputs and error analysis

A strong portfolio keeps inspectable evidence. The cells below profile compact result tables and automatically detect prediction-like columns for residual or misclassification analysis.


In [ ]:
# Load compact result/evidence tables
result_tables = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in sorted(base.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.csv','.tsv','.parquet'} and path.stat().st_size < 25_000_000:
            try:
                if path.suffix.lower() == '.parquet':
                    table = pd.read_parquet(path)
                else:
                    table = pd.read_csv(path, sep='	' if path.suffix.lower() == '.tsv' else ',')
            except Exception as exc:
                print('Could not read', path.name, '-', exc)
                continue
            result_tables.append((path, table))
            print('
RESULT TABLE:', path.relative_to(ROOT) if ROOT in path.parents else path)
            print('shape=', table.shape)
            display(table.head(15))
            numeric = table.select_dtypes(include=np.number).columns.tolist()[:12]
            if numeric:
                display(table[numeric].describe().T.round(4))
print('Inspectable result tables:', len(result_tables))


In [ ]:
# Automatic regression/classification-style error diagnostics
actual_tokens = ('actual','target','truth','y_true','observed','label')
pred_tokens = ('prediction','predicted','forecast','y_pred')
confidence_tokens = ('confidence','probability','proba','risk','uncertainty')
for path, table in result_tables:
    actual_cols = [c for c in table.columns if any(token in str(c).lower() for token in actual_tokens)]
    pred_cols = [c for c in table.columns if any(token in str(c).lower() for token in pred_tokens)]
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in confidence_tokens)]
    if actual_cols and pred_cols and len(table):
        actual_col = actual_cols[0]
        pred_col = next((c for c in pred_cols if c != actual_col), pred_cols[0])
        actual_num = pd.to_numeric(table[actual_col], errors='coerce')
        pred_num = pd.to_numeric(table[pred_col], errors='coerce')
        numeric_mask = actual_num.notna() & pred_num.notna()
        if numeric_mask.sum() >= 10:
            residual = actual_num[numeric_mask] - pred_num[numeric_mask]
            abs_error = residual.abs()
            print('
', path.name, '| MAE=', round(float(abs_error.mean()),5), '| RMSE=', round(float(np.sqrt(np.mean(residual**2))),5), '| bias=', round(float(residual.mean()),5))
            plt.figure(figsize=(7,5))
            plt.scatter(actual_num[numeric_mask], pred_num[numeric_mask], alpha=0.35, s=18)
            lo = min(actual_num[numeric_mask].min(), pred_num[numeric_mask].min())
            hi = max(actual_num[numeric_mask].max(), pred_num[numeric_mask].max())
            plt.plot([lo,hi],[lo,hi], linestyle='--')
            plt.xlabel(str(actual_col))
            plt.ylabel(str(pred_col))
            plt.title(f'Actual vs predicted — {path.name}')
            plt.tight_layout()
            plt.show()
            plt.figure(figsize=(7,4))
            plt.hist(residual, bins=30, alpha=0.82)
            plt.axvline(0, linestyle='--')
            plt.title(f'Residual distribution — {path.name}')
            plt.tight_layout()
            plt.show()
            worst_idx = abs_error.nlargest(min(15,len(abs_error))).index
            cols = list(dict.fromkeys([actual_col,pred_col]+conf_cols[:2]))
            worst = table.loc[worst_idx, cols].copy()
            worst['absolute_error'] = abs_error.loc[worst_idx].values
            display(worst.sort_values('absolute_error', ascending=False))
        else:
            agreement = table[actual_col].astype(str) == table[pred_col].astype(str)
            print('
', path.name, '| classification agreement=', round(float(agreement.mean()),4))
            if (~agreement).any():
                display(table.loc[~agreement, [actual_col,pred_col]+conf_cols[:2]].head(20))
    elif conf_cols:
        for col in conf_cols[:2]:
            values = pd.to_numeric(table[col], errors='coerce').dropna()
            if len(values) >= 10:
                plt.figure(figsize=(7,4))
                plt.hist(values, bins=30, alpha=0.82)
                plt.title(f'{col} distribution — {path.name}')
                plt.tight_layout()
                plt.show()


In [ ]:
# Display retained visual evidence from actual project runs
png_files = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if base.exists():
        png_files.extend(sorted(base.rglob('*.png')))
print('Retained PNG figures:', len(png_files))
for path in png_files[:12]:
    try:
        image = plt.imread(path)
        plt.figure(figsize=(10,6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(str(path.relative_to(ROOT)) if ROOT in path.parents else path.name)
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print('Could not display', path.name, '-', exc)


In [ ]:
# Reproducibility and evidence checklist
checks = [
    {'check':'README present', 'status':(PROJECT/'README.md').exists()},
    {'check':'Recruiter notebook present', 'status':(PROJECT/'project_notebook.ipynb').exists()},
    {'check':'Python implementation present', 'status':any(PROJECT.rglob('*.py'))},
    {'check':'Tests present', 'status':(PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))},
    {'check':'Result/evidence files present', 'status':bool(candidate_files)},
    {'check':'Machine-readable JSON evidence', 'status':bool(json_files)},
    {'check':'Retained visual evidence', 'status':bool(png_files)},
]
checklist = pd.DataFrame(checks)
display(checklist)
print('Evidence checklist pass rate:', f"{100*checklist['status'].mean():.1f}%")
print('A failed item is a prompt to strengthen the project, not something to hide.')


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `run.py`


In [ ]:
from __future__ import annotations

import json
import re
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

ROOT = Path(__file__).resolve().parent
RESULTS = ROOT / "results"
ARTIFACTS = ROOT / "artifacts"
RESULTS.mkdir(exist_ok=True)
ARTIFACTS.mkdir(exist_ok=True)
RANDOM_STATE = 42


@dataclass
class Metrics:
    accuracy: float
    macro_f1: float
    weighted_f1: float
    log_loss: float | None
    auto_route_rate: float
    auto_route_accuracy: float | None


def normalise_text(text: str) -> str:
    text = str(text).replace("\x00", " ")
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.\w+\b", " EMAIL ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def load_corpus():
    kwargs = {
        "remove": ("headers", "footers", "quotes"),
        "shuffle": True,
        "random_state": RANDOM_STATE,
    }
    train = fetch_20newsgroups(subset="train", **kwargs)
    test = fetch_20newsgroups(subset="test", **kwargs)
    x_train = [normalise_text(text) for text in train.data]
    x_test = [normalise_text(text) for text in test.data]
    y_train = np.asarray(train.target, dtype=int)
    y_test = np.asarray(test.target, dtype=int)
    target_names = [str(name) for name in train.target_names]
    return x_train, y_train, x_test, y_test, target_names


def corpus_audit(texts: list[str], labels: np.ndarray, target_names: list[str]) -> dict[str, Any]:
    lengths = np.asarray([len(text.split()) for text in texts], dtype=int)
    empty = int(sum(not text.strip() for text in texts))
    unique_labels, counts = np.unique(labels, return_counts=True)
    if len(unique_labels) != len(target_names):
        raise ValueError("Target-name count does not match observed labels")
    if empty / max(len(texts), 1) > 0.10:
        raise ValueError("Unexpectedly high empty-document rate")
    return {
        "documents": int(len(texts)),
        "classes": int(len(target_names)),
        "empty_documents": empty,
        "median_words": float(np.median(lengths)),
        "p95_words": float(np.percentile(lengths, 95)),
        "class_counts": {target_names[int(k)]: int(v) for k, v in zip(unique_labels, counts)},
    }


def build_vectorizer() -> TfidfVectorizer:
    return TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words="english",
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.98,
        sublinear_tf=True,
        max_features=120000,
        dtype=np.float32,
    )


def build_nb_baseline() -> Pipeline:
    return Pipeline(
        [
            ("tfidf", build_vectorizer()),
            ("model", MultinomialNB(alpha=0.08)),
        ]
    )


def build_calibrated_svm() -> Pipeline:
    svm = LinearSVC(C=2.0, class_weight=None, random_state=RANDOM_STATE)
    calibrated = CalibratedClassifierCV(svm, method="sigmoid", cv=3)
    return Pipeline(
        [
            ("tfidf", build_vectorizer()),
            ("model", calibrated),
        ]
    )


def confidence_policy(probabilities: np.ndarray, threshold: float = 0.55) -> np.ndarray:
    return probabilities.max(axis=1) >= threshold


def evaluate(
    model: Pipeline,
    texts: list[str],
    truth: np.ndarray,
    threshold: float = 0.55,
) -> tuple[Metrics, dict[str, Any], np.ndarray, np.ndarray]:
    prediction = model.predict(texts)
    probabilities = model.predict_proba(texts) if hasattr(model, "predict_proba") else None
    accepted = None
    auto_rate = 1.0
    auto_accuracy = float(accuracy_score(truth, prediction))
    ll = None
    if probabilities is not None:
        accepted = confidence_policy(probabilities, threshold)
        auto_rate = float(accepted.mean())
        auto_accuracy = float(accuracy_score(truth[accepted], prediction[accepted])) if accepted.any() else None
        ll = float(log_loss(truth, probabilities, labels=np.arange(probabilities.shape[1])))
    metrics = Metrics(
        accuracy=float(accuracy_score(truth, prediction)),
        macro_f1=float(f1_score(truth, prediction, average="macro")),
        weighted_f1=float(f1_score(truth, prediction, average="weighted")),
        log_loss=ll,
        auto_route_rate=auto_rate,
        auto_route_accuracy=auto_accuracy,
    )
    detail = {
        "confusion_matrix": confusion_matrix(truth, prediction).tolist(),
        "classification_report": classification_report(truth, prediction, output_dict=True, zero_division=0),
    }
    if probabilities is None:
        probabilities = np.zeros((len(texts), 1), dtype=float)
    if accepted is None:
        accepted = np.ones(len(texts), dtype=bool)
    return metrics, detail, prediction, probabilities


def threshold_sweep(probabilities: np.ndarray, truth: np.ndarray, prediction: np.ndarray) -> pd.DataFrame:
    rows = []
    for threshold in np.arange(0.35, 0.91, 0.05):
        accepted = probabilities.max(axis=1) >= threshold
        rows.append(
            {
                "threshold": float(round(threshold, 2)),
                "auto_route_rate": float(accepted.mean()),
                "auto_route_accuracy": float(accuracy_score(truth[accepted], prediction[accepted])) if accepted.any() else np.nan,
                "review_rate": float((~accepted).mean()),
                "review_rows": int((~accepted).sum()),
            }
        )
    return pd.DataFrame(rows)


def confusion_pairs(confusion: np.ndarray, target_names: list[str], top_n: int = 20) -> pd.DataFrame:
    rows = []
    matrix = confusion.copy().astype(int)
    np.fill_diagonal(matrix, 0)
    for true_index in range(matrix.shape[0]):
        for pred_index in range(matrix.shape[1]):
            count = int(matrix[true_index, pred_index])
            if count:
                rows.append(
                    {
                        "true_category": target_names[true_index],
                        "predicted_category": target_names[pred_index],
                        "count": count,
                    }
                )
    return pd.DataFrame(rows).sort_values("count", ascending=False).head(top_n)


def category_keywords(model: Pipeline, target_names: list[str], top_n: int = 15) -> dict[str, list[str]]:
    vectorizer: TfidfVectorizer = model.named_steps["tfidf"]
    calibrated: CalibratedClassifierCV = model.named_steps["model"]
    feature_names = np.asarray(vectorizer.get_feature_names_out())
    coefficient_sets: list[np.ndarray] = []
    for calibrated_classifier in calibrated.calibrated_classifiers_:
        estimator = getattr(calibrated_classifier, "estimator", None)
        if estimator is None:
            estimator = getattr(calibrated_classifier, "base_estimator", None)
        if estimator is not None and hasattr(estimator, "coef_"):
            coefficient_sets.append(estimator.coef_)
    if not coefficient_sets:
        return {name: [] for name in target_names}
    coefficients = np.mean(np.stack(coefficient_sets, axis=0), axis=0)
    if coefficients.shape[0] != len(target_names):
        return {name: [] for name in target_names}
    insights: dict[str, list[str]] = {}
    for index, name in enumerate(target_names):
        top_indices = np.argsort(coefficients[index])[-top_n:][::-1]
        insights[name] = feature_names[top_indices].tolist()
    return insights


def length_error_slices(texts: list[str], truth: np.ndarray, prediction: np.ndarray, probabilities: np.ndarray) -> pd.DataFrame:
    frame = pd.DataFrame(
        {
            "words": [len(text.split()) for text in texts],
            "target": truth,
            "prediction": prediction,
            "confidence": probabilities.max(axis=1),
        }
    )
    frame["correct"] = (frame["target"] == frame["prediction"]).astype(int)
    frame["length_band"] = pd.cut(
        frame["words"],
        bins=[-1, 10, 50, 150, 500, np.inf],
        labels=["very_short", "short", "medium", "long", "very_long"],
    )
    return (
        frame.groupby("length_band", observed=True)
        .agg(rows=("target", "size"), accuracy=("correct", "mean"), mean_confidence=("confidence", "mean"))
        .reset_index()
    )


def predict_document(model: Pipeline, text: str, target_names: list[str], threshold: float = 0.55) -> dict[str, Any]:
    cleaned = normalise_text(text)
    probabilities = model.predict_proba([cleaned])[0]
    label_index = int(np.argmax(probabilities))
    confidence = float(probabilities[label_index])
    top_indices = np.argsort(probabilities)[-3:][::-1]
    return {
        "predicted_category": target_names[label_index],
        "confidence": confidence,
        "decision": "auto_route" if confidence >= threshold else "human_review",
        "top_categories": [
            {"category": target_names[int(i)], "probability": float(probabilities[int(i)])}
            for i in top_indices
        ],
        "cleaned_preview": cleaned[:300],
    }


def save_json(path: Path, payload: Any) -> None:
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def main() -> None:
    x_train, y_train, x_test, y_test, target_names = load_corpus()
    audit_train = corpus_audit(x_train, y_train, target_names)
    audit_test = corpus_audit(x_test, y_test, target_names)

    baseline = build_nb_baseline()
    baseline.fit(x_train, y_train)
    baseline_metrics, _, _, _ = evaluate(baseline, x_test, y_test, threshold=0.55)

    model = build_calibrated_svm()
    model.fit(x_train, y_train)
    model_metrics, detail, prediction, probabilities = evaluate(model, x_test, y_test, threshold=0.55)

    threshold_table = threshold_sweep(probabilities, y_test, prediction)
    confusion = np.asarray(detail["confusion_matrix"], dtype=int)
    confusion_table = confusion_pairs(confusion, target_names)
    keywords = category_keywords(model, target_names)
    length_slices = length_error_slices(x_test, y_test, prediction, probabilities)

    threshold_table.to_csv(RESULTS / "confidence_thresholds.csv", index=False)
    confusion_table.to_csv(RESULTS / "top_confusions.csv", index=False)
    length_slices.to_csv(RESULTS / "length_error_slices.csv", index=False)
    save_json(RESULTS / "category_keywords.json", keywords)
    joblib.dump({"model": model, "target_names": target_names}, ARTIFACTS / "nlp_document_router.joblib")

    bundle = joblib.load(ARTIFACTS / "nlp_document_router.joblib")
    parity_original = model.predict(x_test[:20])
    parity_loaded = bundle["model"].predict(x_test[:20])
    if not np.array_equal(parity_original, parity_loaded):
        raise RuntimeError("Saved NLP pipeline parity failed")

    example_text = "The graphics driver crashes when I render a 3D scene and the display becomes corrupted."
    payload = {
        "train_audit": audit_train,
        "test_audit": audit_test,
        "naive_bayes_baseline": asdict(baseline_metrics),
        "calibrated_linear_svm": asdict(model_metrics),
        "macro_f1_gain": float(model_metrics.macro_f1 - baseline_metrics.macro_f1),
        "example_prediction": predict_document(model, example_text, target_names),
        "top_confusions": confusion_table.to_dict(orient="records"),
        "limitations": [
            "Historical public benchmark rather than a current enterprise document stream.",
            "Category taxonomy is fixed and does not represent every real routing problem.",
            "Production use requires privacy controls, domain labels, drift monitoring and human escalation for novel text.",
        ],
    }
    save_json(RESULTS / "metrics.json", payload)
    print(json.dumps(payload, indent=2, default=str))


if __name__ == "__main__":
    main()


## Canonical source: `tests/test_nlp.py`


In [ ]:
from pathlib import Path
import sys

import numpy as np

PROJECT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT))

from run import confidence_policy, normalise_text


def test_text_normalisation_redacts_common_contact_patterns():
    text = "Email me at person@example.com and see https://example.com/page"
    cleaned = normalise_text(text)
    assert "person@example.com" not in cleaned
    assert "https://example.com/page" not in cleaned
    assert "EMAIL" in cleaned
    assert "URL" in cleaned


def test_confidence_policy_routes_uncertain_rows_to_review():
    probabilities = np.array([[0.9, 0.1], [0.52, 0.48], [0.2, 0.8]])
    accepted = confidence_policy(probabilities, threshold=0.60)
    assert accepted.tolist() == [True, False, True]


# Portfolio depth check

**Meaningful code lines visible in this notebook:** 448. For a major recruiter-facing application the working target is roughly **1,000 meaningful lines**, with a practical guide of about 600–1,400 depending on the problem. This notebook is below the major-project guide and should grow only through substantive analysis/application depth.

Line count is not a quality metric by itself. Add code only when it improves the real project: data acquisition, validation, cleaning, EDA, visualisation, feature engineering, baselines, model comparison, tuning, leakage control, error analysis, explainability, uncertainty, inference, tests, monitoring, deployment or decision logic.
